<!-- paired-role-banner -->
> 🟧 **실습 노트북 02 · 빈 코드 창**
>
> 오른쪽의 같은 이름 **정답 노트북**과 셀 위치를 맞춰 보세요. 먼저 정답을 가리고 직접
> 작성한 뒤 막힐 때 한 줄씩 비교하는 방식을 권장합니다. `NotImplementedError`가 있는 셀을
> 완성하면 됩니다.

# 02. NumPy: 벡터화, broadcasting, 수치 안정성, 선형대수

딥러닝 프레임워크의 핵심 연산은 NumPy와 같은 배열 사고방식에서 출발합니다.
이 노트북에서는 shape·axis·dtype을 추적하면서 **벡터화, broadcasting, 난수 생성기,
안정적인 softmax/cross-entropy, 행렬곱, cosine similarity, 최소제곱, SVD**를 연습합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=sys.path.insert -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>sys.path.insert(index, path)</code></summary>

#### `sys.path.insert(index, path)`

- **역할:** Python 모듈 검색 경로의 지정 위치에 경로를 추가합니다.
- **입력·반환:** index와 경로 문자열을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 노트북에서 프로젝트 `src` 모듈을 import할 수 있게 합니다.
- **주의점:** 전역 import 상태를 바꾸므로 중복·우선순위와 실행 위치에 주의합니다.
- **공식 문서:** [sys.path.insert](https://docs.python.org/3/library/sys.html#sys.path)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """현재 폴더부터 위로 올라가며 프로젝트 루트를 찾습니다."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "pyproject.toml과 src/가 있는 프로젝트 폴더를 찾지 못했습니다."
    )


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("project:", PROJECT_ROOT)
print("python :", sys.executable)

In [ ]:
from time import perf_counter

import numpy as np

np.set_printoptions(precision=4, suppress=True)

## 1. 배열의 기본 계약: shape, dtype, ndim

AI 코드에서 가장 흔한 오류는 값 자체보다 shape의 의미를 잃는 것입니다.
아래 배열은 `(batch, features)` 구조입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.array -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.array(object, dtype=None, copy=True, ...)</code></summary>

#### `np.array(object, dtype=None, copy=True, ...)`

- **역할:** 입력 데이터로 새 NumPy 배열을 만듭니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.array](https://numpy.org/doc/stable/reference/generated/numpy.array.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
features = np.array(
    [
        [2.0, 0.5, 1.0],
        [1.0, 1.5, 0.0],
        [3.0, 0.2, 1.0],
        [0.5, 2.0, 0.0],
    ],
    dtype=np.float32,
)

assert features.shape == (4, 3)
assert features.ndim == 2
assert features.dtype == np.float32
print("shape:", features.shape, "dtype:", features.dtype)
features

## 2. Python loop와 벡터화

벡터화는 연산 의도를 배열 단위로 표현해 C/BLAS 수준 구현을 활용합니다.
성능 측정은 환경에 따라 달라지므로 속도를 assert하지 않고 결과의 동등성만 검증합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=perf_counter,np.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>perf_counter()</code></summary>

#### `perf_counter()`

- **역할:** 짧은 구간 측정에 적합한 고해상도 단조 시계를 읽습니다.
- **입력·반환:** 인자 없이 호출하고 초 단위 부동소수점 값을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습·추론 latency의 시작과 끝 차이를 측정합니다.
- **주의점:** 절댓값에는 의미가 없고 GPU 시간은 실행 동기화 뒤 측정해야 합니다.
- **공식 문서:** [perf_counter](https://docs.python.org/3/library/time.html#time.perf_counter)

</details>

<details>
<summary><code>np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 배열의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 배열 `a`, 기준 배열 `b`, 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 부동소수점 계산 결과를 정확한 값이 아니라 수치적으로 같은지 확인합니다.
- **주의점:** 조건은 `abs(a-b) <= atol + rtol*abs(b)`입니다. `==`와 달리 오차를 허용하고, `b`가 기준이라 비대칭일 수 있으며 1보다 매우 작은 값에는 기본 `atol`이 너무 클 수 있습니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [np.allclose](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
values = np.linspace(-3, 3, 200_000, dtype=np.float64)

started = perf_counter()
loop_result = np.array([value * value + 2 * value + 1 for value in values])
loop_ms = (perf_counter() - started) * 1_000

started = perf_counter()
vectorized_result = values**2 + 2 * values + 1
vectorized_ms = (perf_counter() - started) * 1_000

assert np.allclose(loop_result, vectorized_result)
print(f"loop={loop_ms:.2f} ms, vectorized={vectorized_ms:.2f} ms")

## 3. Broadcasting

`(batch, features)` 배열에서 `(features,)` 평균을 빼면 NumPy가 마지막 축을 맞춰 확장합니다.
실제 복사가 아니라 호환 가능한 shape 규칙이 적용됩니다.

## 4. `axis`와 `keepdims`

`axis=0`은 batch를 모으고 feature별 통계를, `axis=1`은 feature를 모으고 샘플별 통계를 냅니다.
`keepdims=True`는 축을 길이 1로 유지해 후속 broadcasting을 명확하게 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
feature_mean = features.mean(axis=0)  # (features,)
centered = features - feature_mean  # (batch, features) - (features,)

assert feature_mean.shape == (3,)
assert centered.shape == features.shape
assert np.allclose(centered.mean(axis=0), 0.0, atol=1e-6)
centered

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.sum,np.where -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

<details>
<summary><code>np.where(condition, x, y)</code></summary>

#### `np.where(condition, x, y)`

- **역할:** 조건에 따라 원소를 선택합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.where](https://numpy.org/doc/stable/reference/generated/numpy.where.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
row_sums = features.sum(axis=1, keepdims=True)
normalized_rows = features / np.where(row_sums == 0, 1, row_sums)

assert row_sums.shape == (4, 1)
assert np.allclose(normalized_rows.sum(axis=1), 1.0)
normalized_rows

## 5. Boolean mask와 fancy indexing

조건을 배열로 만들고 필요한 샘플만 선택합니다. 원본 순서를 보존하지만 대부분 새 배열을 만듭니다.

## 6. 전역 난수 대신 `Generator` 사용

함수 내부에 독립적인 `Generator`를 전달하면 실험 간 난수 상태가 덜 얽힙니다.
동일한 seed로 만든 두 generator는 같은 수열을 생성합니다.

In [ ]:
confidence = np.array([0.92, 0.41, 0.77, 0.33])
review_mask = confidence < 0.6
review_rows = features[review_mask]

assert review_mask.tolist() == [False, True, False, True]
assert review_rows.shape == (2, 3)
review_rows

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.random.default_rng,np.array_equal -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.random.default_rng(seed=None)</code></summary>

#### `np.random.default_rng(seed=None)`

- **역할:** 독립적인 NumPy 난수 생성기 `Generator`를 만듭니다.
- **입력·반환:** 선택적 시드를 받고 `Generator` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 전역 상태를 오염시키지 않고 데이터나 실험 난수를 재현합니다.
- **주의점:** 생성기를 다시 만들지 않으면 호출할 때마다 내부 상태가 진행됩니다.
- **공식 문서:** [np.random.default_rng](https://numpy.org/doc/stable/reference/random/generator.html)

</details>

<details>
<summary><code>np.array_equal(a1, a2, equal_nan=False)</code></summary>

#### `np.array_equal(a1, a2, equal_nan=False)`

- **역할:** shape과 모든 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 배열을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 정수 인덱스나 마스크처럼 오차 허용이 없어야 하는 결과를 확인합니다.
- **주의점:** 부동소수점 연산 결과에는 보통 `allclose`가 더 적합합니다.
- **공식 문서:** [np.array_equal](https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
rng_a = np.random.default_rng(42)
rng_b = np.random.default_rng(42)

sample_a = rng_a.normal(loc=0.0, scale=1.0, size=(2, 4))
sample_b = rng_b.normal(loc=0.0, scale=1.0, size=(2, 4))

assert np.array_equal(sample_a, sample_b)
sample_a

## 7. 수치적으로 안정적인 softmax

`exp(1000)`은 overflow합니다. 각 행의 최댓값을 먼저 빼도 softmax 결과는 같고 지수 범위는 안전해집니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.asarray,np.exp,np.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.asarray(a, dtype=None, order=None)</code></summary>

#### `np.asarray(a, dtype=None, order=None)`

- **역할:** 가능하면 복사 없이 입력을 NumPy 배열로 해석합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.asarray](https://numpy.org/doc/stable/reference/generated/numpy.asarray.html)

</details>

<details>
<summary><code>np.exp(x)</code></summary>

#### `np.exp(x)`

- **역할:** 원소별 자연지수 함수를 계산합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=e^{x_i}
$$

- **기호:** $e$는 자연로그의 밑입니다.
- **수식 → 코드:** API가 각 원소를 양수 값으로 지수 변환합니다.
- **직관:** 더하기 차이를 곱셈 비율로 바꾸며 log-domain 값을 원래 scale로 복원합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [np.exp](https://numpy.org/doc/stable/reference/generated/numpy.exp.html)

</details>

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 1: 7. 수치적으로 안정적인 softmax
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("02번 실습의 TODO 1을 완성하세요")

## 8. log-sum-exp와 cross-entropy

정답 클래스의 `-log(probability)`가 cross-entropy입니다. 확률을 먼저 계산한 뒤 log를 취하는 대신
logits 공간의 log-sum-exp를 사용하면 극단적인 값에서도 안정적입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.log,np.isclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.log(x)</code></summary>

#### `np.log(x)`

- **역할:** 원소별 자연로그를 계산합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\log x_i,\qquad x_i>0
$$

- **기호:** $\log$는 자연로그입니다.
- **수식 → 코드:** API가 곱셈 관계를 덧셈 관계로 바꾸고 작은 확률의 크기를 안정적으로 표현합니다.
- **직관:** 확률 곱을 log-probability 합으로 계산할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같으며 입력은 양수여야 합니다.
- **공식 문서:** [np.log](https://numpy.org/doc/stable/reference/generated/numpy.log.html)

</details>

<details>
<summary><code>np.isclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `np.isclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 각 원소 쌍이 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 브로드캐스트 가능한 두 배열을 받고 원소별 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 어느 위치에서 수치 오차가 생겼는지 원소 단위로 확인합니다.
- **주의점:** `allclose`와 달리 결과가 배열이며 같은 비대칭 오차 공식을 사용합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [np.isclose](https://numpy.org/doc/stable/reference/generated/numpy.isclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 2: 8. log-sum-exp와 cross-entropy
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("02번 실습의 TODO 2을 완성하세요")

## 9. 행렬곱: batch 입력과 선형 계층

선형 계층은 `X @ W + b`입니다. `(batch, input_dim) @ (input_dim, output_dim)`의 결과는
`(batch, output_dim)`이 됩니다.

## 10. Cosine similarity

임베딩 검색은 방향이 비슷한 벡터를 찾습니다. 0 벡터의 norm은 작은 epsilon으로 보호합니다.

In [ ]:
rng = np.random.default_rng(7)
weights = rng.normal(0, 0.1, size=(3, 2))
bias = np.zeros(2)
logits = features @ weights + bias

assert logits.shape == (4, 2)
logits

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.linalg.norm -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.linalg.norm(x, ord=None, axis=None, ...)</code></summary>

#### `np.linalg.norm(x, ord=None, axis=None, ...)`

- **역할:** 벡터나 행렬의 norm을 계산합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\lVert x\rVert_p=\left(\sum_i\lvert x_i\rvert^p\right)^{1/p},\qquad \lVert x\rVert_2=\sqrt{\sum_i x_i^2}
$$

- **기호:** $p$는 norm 차수이며 $p=2$가 유클리드 길이입니다.
- **수식 → 코드:** `ord`와 `dim`·`axis`가 어떤 원소 묶음의 크기를 계산할지 정합니다.
- **직관:** 여러 성분을 하나의 거리·크기 값으로 요약합니다.
- **shape 확인:** 지정한 축은 결과에서 제거되며 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [np.linalg.norm](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 3: 9. 행렬곱: batch 입력과 선형 계층
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("02번 실습의 TODO 3을 완성하세요")

## 11. 최소제곱으로 선형 회귀 풀기

역행렬을 직접 계산하기보다 `np.linalg.lstsq`를 사용합니다. rank가 부족한 경우에도 더 안정적입니다.

## 12. SVD와 저차원 표현

중심화한 행렬을 SVD로 분해하면 주성분 방향을 얻을 수 있습니다.
여기서는 첫 두 방향으로 투영하고 shape과 복원 오차를 확인합니다.

In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y = 1.5 * x + 0.7
design = np.column_stack([x, np.ones_like(x)])

coefficients, residuals, rank, singular_values = np.linalg.lstsq(design, y, rcond=None)
slope, intercept = coefficients

assert np.isclose(slope, 1.5)
assert np.isclose(intercept, 0.7)
print({"slope": slope, "intercept": intercept, "rank": rank})

In [ ]:
centered64 = features.astype(np.float64) - features.mean(axis=0)
u, singular_values, vt = np.linalg.svd(centered64, full_matrices=False)
projected_2d = centered64 @ vt[:2].T
reconstructed = projected_2d @ vt[:2]

assert projected_2d.shape == (4, 2)
assert np.linalg.norm(centered64 - reconstructed) >= 0
print("singular values:", singular_values)
projected_2d

## 13. View와 copy

slice는 원본 메모리를 공유할 수 있습니다. 전처리 중 원본을 보존해야 한다면 `.copy()`를 명시하세요.

In [ ]:
original = np.arange(8)
view = original[2:5]
independent = original[2:5].copy()

view[0] = 999
independent[1] = -1

assert original[2] == 999  # view 수정이 원본에 반영됨
assert original[3] != -1  # copy 수정은 원본과 독립적
original, view, independent

## TODO 1: feature 표준화 함수

구현을 지운 뒤 `mean=0`, `std=1`이 되도록 다시 작성하세요. 분산이 0인 열은 0으로 유지해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.std -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.std(dim=None, correction=1, keepdim=False)</code></summary>

#### `tensor.std(dim=None, correction=1, keepdim=False)`

- **역할:** 지정 축의 표준편차를 계산합니다.
- **입력·반환:** 텐서와 축·보정값을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표현의 퍼짐, 데이터 scale, 학습 안정성을 진단합니다.
- **주의점:** 표본 수와 `correction`에 따라 작은 batch에서는 NaN이 될 수 있습니다.

##### 관련 수식과 코드 연결

$$
s=\sqrt{\frac{1}{N-\delta}\sum_{i=1}^{N}(x_i-\bar x)^2}
$$

- **기호:** $\delta$는 correction이며 NumPy 기본은 0, PyTorch `std` 기본은 1입니다.
- **수식 → 코드:** API의 `correction` 또는 `ddof`가 분모를 $N$과 $N-1$ 중 무엇으로 할지 정합니다.
- **직관:** 평균 주변에 값이 얼마나 퍼져 있는지를 원래 단위로 나타냅니다.
- **shape 확인:** 평균과 마찬가지로 지정한 축이 제거되거나 크기 1로 유지됩니다.
- **공식 문서:** [Tensor.std](https://docs.pytorch.org/docs/stable/generated/torch.std.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 4: TODO 1: feature 표준화 함수
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("02번 실습의 TODO 4을 완성하세요")

## TODO 2: batch query-document 유사도

loop 없이 `(queries, dim) @ (documents, dim).T`를 사용해 모든 조합의 cosine similarity를 계산하세요.

In [ ]:
def pairwise_cosine(
    queries: np.ndarray, documents: np.ndarray, eps: float = 1e-12
) -> np.ndarray:
    # TODO: 각 행을 L2 정규화한 뒤 행렬곱하세요.
    query_norms = np.maximum(np.linalg.norm(queries, axis=1, keepdims=True), eps)
    document_norms = np.maximum(np.linalg.norm(documents, axis=1, keepdims=True), eps)
    return (queries / query_norms) @ (documents / document_norms).T


pairwise = pairwise_cosine(features[:2], features)
assert pairwise.shape == (2, 4)
assert np.allclose(np.diag(pairwise[:, :2]), 1.0)
pairwise

## 정리

- 항상 `(batch, sequence, hidden)`처럼 각 축의 의미를 말로 적습니다.
- 벡터화는 속도뿐 아니라 코드의 수학적 의도를 선명하게 합니다.
- softmax/log 연산은 overflow와 underflow를 고려합니다.
- 역행렬보다 `solve`, `lstsq`, SVD 같은 안정적인 도구를 우선합니다.
- 다음 노트북에서는 같은 원칙을 표 형태 데이터와 데이터 누수 문제에 적용합니다.